In [14]:
import tensorflow as tf
import tensorflow_hub as hub
import librosa

audio_model = hub.load('https://tfhub.dev/google/yamnet/1')

label_map_path = audio_model.class_map_path().numpy().decode('utf-8')
sound_labels = [
    line.strip().split(',')[2]
    for line in tf.io.gfile.GFile(label_map_path)
    if ',' in line and not line.startswith('index')
]

city_sound_categories = {
    "blockage_honk": ["Toot"],
    "emergency_siren": ["Siren", "Police car (siren)", "Ambulance (siren)"],
    "crash": ["Breaking"],
}

detection_thresholds = {
    "blockage_honk": 0.12,
    "emergency_siren": 0.15,
    "crash": 0.10,
}

def analyze_audio_clip(audio_clip_path):
    audio_waveform, sample_rate = librosa.load(audio_clip_path, sr=16000)
    audio_waveform = tf.convert_to_tensor(audio_waveform, dtype=tf.float32)

    probabilities, embeddings, spectrogram = audio_model(audio_waveform)

    print("\nDebugging Frame Probabilities:")
    for frame_index, frame_scores in enumerate(probabilities.numpy()):
        print(f"Frame {frame_index}:")
        for label_idx, probability in enumerate(frame_scores):
            if probability > 0.1:
                print(f"  - {sound_labels[label_idx]}: {probability:.4f}")

    sound_events = identify_city_sounds(probabilities)

    if sound_events["alerts"]:
        print("\n*** ALERT: CITY SOUND EVENTS DETECTED ***")
        for alert in sound_events["alerts"]:
            print(f"- Sound Category: {alert}")
        for detail in sound_events["details"]:
            print(
                f"  - Detected Sound: {detail['sound']}, "
                f"Confidence: {detail['confidence']:.2f}, Frame: {detail['frame_idx']}"
            )
    else:
        print("\nNo significant city-related sounds detected.")

def identify_city_sounds(probabilities):
    detected_events = {"alerts": [], "details": []}
    frame_probabilities = probabilities.numpy()

    for frame_index, frame_scores in enumerate(frame_probabilities):
        for category, sound_keywords in city_sound_categories.items():
            for keyword in sound_keywords:
                try:
                    label_idx = sound_labels.index(keyword)
                except ValueError:
                    print(f"Keyword '{keyword}' not found in sound_labels.")
                    continue

                print(
                    f"Analyzing '{keyword}' at index {label_idx} in frame {frame_index} "
                    f"with confidence {frame_scores[label_idx]:.4f} (Threshold: {detection_thresholds[category]:.4f})"
                )

                if frame_scores[label_idx] >= detection_thresholds[category]:
                    if category not in detected_events["alerts"]:
                        detected_events["alerts"].append(category)
                    detected_events["details"].append({
                        "sound": keyword,
                        "confidence": frame_scores[label_idx],
                        "frame_idx": frame_index,
                    })
                    print(
                        f"DETECTED: {category} caused by '{keyword}' "
                        f"with confidence {frame_scores[label_idx]:.4f} at frame {frame_index}"
                    )
    return detected_events

audio_path = "../inputs/police-siren.mp3"
analyze_audio_clip(audio_path)


Debugging Frame Probabilities:
Frame 0:
  - Police car (siren): 0.5591
  - Alarm: 0.4474
  - Siren: 0.5585
Frame 1:
  - Police car (siren): 0.4091
  - Alarm: 0.3615
  - Siren: 0.4704
  - Sound effect: 0.1493
Frame 2:
  - Emergency vehicle: 0.1106
  - Police car (siren): 0.3811
  - Ambulance (siren): 0.1323
  - Alarm: 0.5835
  - Siren: 0.5908
  - Sound effect: 0.1147
Frame 3:
  - Police car (siren): 0.3575
  - Alarm: 0.1651
  - Siren: 0.2517
  - Sound effect: 0.2383
Frame 4:
  - Police car (siren): 0.2237
  - Alarm: 0.1283
  - Siren: 0.1621
  - Sound effect: 0.3194
Frame 5:
  - Police car (siren): 0.3083
  - Alarm: 0.1225
  - Siren: 0.1990
  - Sound effect: 0.1923
Frame 6:
  - Police car (siren): 0.1799
  - Alarm: 0.4284
  - Siren: 0.4570
  - Sound effect: 0.2065
Analyzing 'Toot' at index 303 in frame 0 with confidence 0.0000 (Threshold: 0.1200)
Analyzing 'Siren' at index 390 in frame 0 with confidence 0.5585 (Threshold: 0.1500)
DETECTED: emergency_siren caused by 'Siren' with confiden